# Module 4.1: Market Regime Detection - Bull/Bear/Sideways

## Learning Objectives
By completing this notebook, you will:
1. Apply HMMs to detect market regimes in S&P 500 data
2. Identify bull, bear, and sideways market states
3. Analyze regime characteristics and persistence
4. Backtest regime-aware investment strategies
5. Validate regime detection accuracy

## Prerequisites
- Gaussian HMM implementation (Module 3)
- Financial market basics
- Python data analysis (pandas, numpy)

## Estimated Time: 55 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Gaussian HMM implementation (Module 3)",
    "Financial market basics",
    "Python data analysis (pandas, numpy)"
])

In [ ]:
# Setup and imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("Libraries loaded successfully")

## 1. Generate Realistic S&P 500 Data

We'll create synthetic S&P 500 returns with realistic regime characteristics:
- **Bull markets**: Positive drift, moderate volatility, long duration
- **Bear markets**: Negative drift, high volatility, shorter duration  
- **Sideways markets**: Near-zero drift, low volatility, transitional

In [ ]:
# Purpose: Generate realistic market data with regime structure
# Key Concept: Stylized facts of equity returns

def generate_sp500_data(T=2520, start_date='2015-01-01', random_seed=42):
    """
    Generate synthetic S&P 500 returns (~10 years of daily data).
    
    Regime characteristics:
    - Bull: μ=0.08% daily, σ=1.0% (20% annual return, 16% vol)
    - Bear: μ=-0.12% daily, σ=2.5% (-30% annual, 40% vol)
    - Sideways: μ=0.00% daily, σ=0.8% (0% annual, 13% vol)
    """
    np.random.seed(random_seed)
    
    # Regime parameters (daily)
    regimes = {
        0: {'name': 'Bull', 'mu': 0.0008, 'sigma': 0.010},
        1: {'name': 'Bear', 'mu': -0.0012, 'sigma': 0.025},
        2: {'name': 'Sideways', 'mu': 0.0000, 'sigma': 0.008}
    }
    
    # Transition matrix (empirical frequencies from historical data)
    A = np.array([
        [0.97, 0.01, 0.02],  # Bull is very persistent
        [0.05, 0.85, 0.10],  # Bear is sharp but shorter
        [0.15, 0.05, 0.80]   # Sideways transitions to Bull often
    ])
    
    pi = np.array([0.60, 0.10, 0.30])  # Start in Bull more often
    
    # Simulate state sequence
    states = np.zeros(T, dtype=int)
    states[0] = np.random.choice(3, p=pi)
    
    for t in range(1, T):
        states[t] = np.random.choice(3, p=A[states[t-1]])
    
    # Generate returns
    returns = np.zeros(T)
    for t in range(T):
        regime = regimes[states[t]]
        returns[t] = np.random.normal(regime['mu'], regime['sigma'])
    
    # Create price series (start at 100)
    prices = 100 * np.exp(np.cumsum(returns))
    
    # Create date index
    dates = pd.date_range(start=start_date, periods=T, freq='B')  # Business days
    
    # Create DataFrame
    df = pd.DataFrame({
        'Date': dates,
        'Price': prices,
        'Return': returns,
        'True_Regime': [regimes[s]['name'] for s in states],
        'True_State': states
    })
    
    return df

# Generate data
df = generate_sp500_data(T=2520, start_date='2015-01-01')

print("S&P 500 Synthetic Data")
print("="*70)
print(f"Date range: {df['Date'].iloc[0].date()} to {df['Date'].iloc[-1].date()}")
print(f"Trading days: {len(df)}")
print(f"\nPrice statistics:")
print(f"  Initial: ${df['Price'].iloc[0]:.2f}")
print(f"  Final:   ${df['Price'].iloc[-1]:.2f}")
print(f"  Min:     ${df['Price'].min():.2f}")
print(f"  Max:     ${df['Price'].max():.2f}")
print(f"\nReturn statistics (daily):")
print(f"  Mean: {df['Return'].mean():.6f} ({df['Return'].mean()*252*100:.2f}% annualized)")
print(f"  Std:  {df['Return'].std():.6f} ({df['Return'].std()*np.sqrt(252)*100:.2f}% annualized)")
print(f"\nRegime distribution:")
print(df['True_Regime'].value_counts())

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price
ax1 = axes[0]
ax1.plot(df['Date'], df['Price'], 'b-', linewidth=1.5)
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title('S&P 500 Price', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Returns with regime background
ax2 = axes[1]
ax2.bar(df['Date'], df['Return'], width=1, alpha=0.7, edgecolor='none')
ax2.axhline(0, color='red', linestyle='--', alpha=0.3)
ax2.set_ylabel('Daily Return', fontsize=12)
ax2.set_title('Daily Returns', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# True regimes
ax3 = axes[2]
colors = {'Bull': 'green', 'Bear': 'red', 'Sideways': 'gray'}
for i in range(len(df)-1):
    regime = df['True_Regime'].iloc[i]
    ax3.axvspan(df['Date'].iloc[i], df['Date'].iloc[i+1], 
               alpha=0.7, color=colors[regime])
ax3.set_ylabel('Regime', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.set_title('True Market Regimes', fontsize=14, fontweight='bold')
ax3.set_ylim(0, 1)
ax3.set_yticks([])

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[r], label=r) for r in ['Bull', 'Bear', 'Sideways']]
ax3.legend(handles=legend_elements, loc='upper right', ncol=3)

plt.tight_layout()
plt.show()

## 2. Feature Engineering for Regime Detection

Instead of using raw returns, we create features that capture regime characteristics:
- Returns (trend)
- Realized volatility (risk level)
- Momentum (persistence)

In [ ]:
# Purpose: Engineer features that help discriminate regimes
# Key Concept: Volatility and momentum are regime-dependent

def compute_features(df, vol_window=20, mom_window=10):
    """
    Compute regime-informative features.
    
    Features:
    1. Return: Current daily return
    2. Realized volatility: Rolling std of returns
    3. Momentum: Rolling mean return (trend)
    """
    features = pd.DataFrame(index=df.index)
    
    # Feature 1: Return
    features['return'] = df['Return']
    
    # Feature 2: Realized volatility
    features['volatility'] = df['Return'].rolling(vol_window).std()
    
    # Feature 3: Momentum
    features['momentum'] = df['Return'].rolling(mom_window).mean()
    
    # Drop NaN from rolling windows
    features = features.dropna()
    
    return features

# Compute features
features = compute_features(df)

print("Feature Statistics")
print("="*70)
print(features.describe())

# Visualize features by regime
# Align with features (drop first rows)
true_regimes_aligned = df['True_Regime'].iloc[features.index]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(['return', 'volatility', 'momentum']):
    ax = axes[i]
    
    for regime in ['Bull', 'Bear', 'Sideways']:
        mask = true_regimes_aligned == regime
        data = features.loc[mask, feature]
        ax.hist(data, bins=50, alpha=0.6, label=regime)
    
    ax.set_xlabel(feature.capitalize(), fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'{feature.capitalize()} by Regime', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Fit HMM for Regime Detection

We'll use a 3-state Gaussian HMM on the engineered features.

In [ ]:
# Purpose: Fit HMM to detect market regimes
# Key Concept: HMM learns regime characteristics from data

# Simple Gaussian HMM implementation
class GaussianHMM:
    def __init__(self, n_states):
        self.K = n_states
        self.pi = np.ones(self.K) / self.K
        self.A = np.random.rand(self.K, self.K)
        self.A = self.A / self.A.sum(axis=1, keepdims=True)
        self.means = None
        self.stds = None
    
    def _emission_prob(self, obs, state):
        return stats.norm.pdf(obs, self.means[state], self.stds[state])
    
    def fit(self, observations, max_iter=50, tol=1e-4, verbose=True):
        T = len(observations)
        
        # Initialize with k-means-like approach
        quantiles = np.linspace(0, 100, self.K+1)
        self.means = np.array([np.percentile(observations, (quantiles[i]+quantiles[i+1])/2) 
                              for i in range(self.K)])
        self.stds = np.ones(self.K) * observations.std()
        
        log_likelihoods = []
        
        for iteration in range(max_iter):
            # Forward
            alpha = np.zeros((T, self.K))
            scaling = np.zeros(T)
            
            alpha[0] = self.pi * np.array([self._emission_prob(observations[0], i) 
                                          for i in range(self.K)])
            scaling[0] = np.sum(alpha[0])
            alpha[0] /= scaling[0]
            
            for t in range(1, T):
                for j in range(self.K):
                    alpha[t, j] = np.sum(alpha[t-1] * self.A[:, j]) * \
                                  self._emission_prob(observations[t], j)
                scaling[t] = np.sum(alpha[t])
                alpha[t] /= scaling[t]
            
            log_lik = np.sum(np.log(scaling + 1e-300))
            log_likelihoods.append(log_lik)
            
            # Backward
            beta = np.zeros((T, self.K))
            beta[-1] = 1.0 / scaling[-1]
            
            for t in range(T-2, -1, -1):
                for i in range(self.K):
                    beta[t, i] = np.sum(
                        self.A[i, :] *
                        np.array([self._emission_prob(observations[t+1], j) 
                                 for j in range(self.K)]) *
                        beta[t+1, :]
                    )
                beta[t] /= scaling[t]
            
            # Compute gamma and xi
            gamma = alpha * beta
            gamma /= gamma.sum(axis=1, keepdims=True)
            
            xi = np.zeros((T-1, self.K, self.K))
            for t in range(T-1):
                denom = 0.0
                for i in range(self.K):
                    for j in range(self.K):
                        xi[t, i, j] = alpha[t, i] * self.A[i, j] * \
                                      self._emission_prob(observations[t+1], j) * \
                                      beta[t+1, j]
                        denom += xi[t, i, j]
                xi[t] /= (denom + 1e-10)
            
            # M-step
            self.pi = gamma[0]
            
            for i in range(self.K):
                for j in range(self.K):
                    self.A[i, j] = np.sum(xi[:, i, j]) / (np.sum(gamma[:-1, i]) + 1e-10)
            self.A = self.A / self.A.sum(axis=1, keepdims=True)
            
            for i in range(self.K):
                gamma_sum = np.sum(gamma[:, i])
                self.means[i] = np.sum(gamma[:, i] * observations) / gamma_sum
                diff = observations - self.means[i]
                self.stds[i] = np.sqrt(np.sum(gamma[:, i] * diff**2) / gamma_sum)
            
            if verbose and (iteration % 10 == 0 or iteration == max_iter - 1):
                print(f"Iteration {iteration:3d}: log L = {log_lik:.4f}")
            
            if iteration > 0 and abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
                if verbose:
                    print(f"Converged after {iteration+1} iterations")
                break
        
        return log_likelihoods
    
    def predict(self, observations):
        T = len(observations)
        log_delta = np.zeros((T, self.K))
        psi = np.zeros((T, self.K), dtype=int)
        
        for i in range(self.K):
            log_delta[0, i] = np.log(self.pi[i] + 1e-10) + \
                             np.log(self._emission_prob(observations[0], i) + 1e-10)
        
        for t in range(1, T):
            for j in range(self.K):
                probs = log_delta[t-1] + np.log(self.A[:, j] + 1e-10)
                psi[t, j] = np.argmax(probs)
                log_delta[t, j] = probs[psi[t, j]] + \
                                 np.log(self._emission_prob(observations[t], j) + 1e-10)
        
        states = np.zeros(T, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states

# Fit HMM on returns only (univariate)
print("Fitting 3-state Gaussian HMM to Returns")
print("="*70 + "\n")

hmm = GaussianHMM(n_states=3)
returns_array = features['return'].values
log_liks = hmm.fit(returns_array, max_iter=100, verbose=True)

print("\n" + "="*70)
print("LEARNED PARAMETERS")
print("="*70)
print(f"\nMeans (daily): {hmm.means}")
print(f"Stds (daily):  {hmm.stds}")
print(f"\nMeans (annualized %): {hmm.means * 252 * 100}")
print(f"Stds (annualized %):  {hmm.stds * np.sqrt(252) * 100}")
print(f"\nTransition matrix:")
print(hmm.A)

# Decode states
decoded_states = hmm.predict(returns_array)

# Label states by mean
sorted_idx = np.argsort(hmm.means)
label_map = {sorted_idx[2]: 'Bull', sorted_idx[1]: 'Sideways', sorted_idx[0]: 'Bear'}

print("\n" + "="*70)
print("DETECTED REGIMES")
print("="*70)
for state in range(3):
    mask = decoded_states == state
    print(f"\nState {state} ({label_map[state]}):")
    print(f"  Days: {mask.sum()}")
    print(f"  Mean return: {returns_array[mask].mean():.6f}")
    print(f"  Volatility:  {returns_array[mask].std():.6f}")

## Exercise 1: Regime Detection Accuracy

**Task:** Compare detected regimes to true regimes:
1. Compute confusion matrix
2. Calculate accuracy, precision, recall for each regime
3. Visualize agreement between true and detected

**Expected Output:** Performance metrics and visualization.

In [ ]:
# YOUR CODE HERE
# ---------------

def evaluate_regime_detection(true_states, detected_states, label_map):
    """
    Evaluate regime detection performance.
    
    Returns
    -------
    confusion_matrix : ndarray (3, 3)
    accuracy : float
    metrics : dict with precision/recall by regime
    """
    # YOUR IMPLEMENTATION
    return None, None, None

# Align true states with features
true_states_aligned = df['True_State'].iloc[features.index].values

confusion, accuracy, metrics = evaluate_regime_detection(
    true_states_aligned, decoded_states, label_map
)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert confusion is not None, "Don't forget confusion matrix!"
    assert accuracy is not None, "Don't forget accuracy!"
    assert metrics is not None, "Don't forget metrics!"
    
    assert confusion.shape == (3, 3), "Confusion matrix should be 3x3"
    assert 0 <= accuracy <= 1, "Accuracy should be in [0, 1]"
    
    print("✅ Exercise 1 passed!")
    print(f"\nOverall accuracy: {accuracy:.2%}")
    print("\nConfusion Matrix:")
    print(confusion)
    print("\nPer-Regime Metrics:")
    for regime in ['Bull', 'Bear', 'Sideways']:
        print(f"\n{regime}:")
        print(f"  Precision: {metrics[regime]['precision']:.2%}")
        print(f"  Recall: {metrics[regime]['recall']:.2%}")

test_exercise_1()

## Exercise 2: Regime Transition Analysis

**Task:** Analyze regime transition dynamics:
1. Find all detected regime transitions
2. Compute average duration of each regime
3. Identify typical transition patterns
4. Compare to true regime durations

**Expected Output:** Transition statistics and patterns.

In [ ]:
# YOUR CODE HERE
# ---------------

def analyze_regime_transitions(states, label_map):
    """
    Analyze regime transition patterns and durations.
    """
    # YOUR IMPLEMENTATION
    return None

transition_analysis = analyze_regime_transitions(decoded_states, label_map)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert transition_analysis is not None, "Don't forget analysis!"
    
    print("✅ Exercise 2 passed!")
    print("\nRegime Transition Analysis:")
    print(transition_analysis)

test_exercise_2()

## Exercise 3: Simple Regime-Based Strategy

**Task:** Backtest a simple strategy:
- **Bull regime**: 100% stocks
- **Bear regime**: 0% stocks (cash)
- **Sideways**: 50% stocks

Compare cumulative returns to buy-and-hold.

**Expected Output:** Strategy performance comparison.

In [ ]:
# YOUR CODE HERE
# ---------------

def backtest_regime_strategy(returns, states, label_map):
    """
    Backtest regime-based allocation strategy.
    """
    # YOUR IMPLEMENTATION
    return None

strategy_results = backtest_regime_strategy(returns_array, decoded_states, label_map)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert strategy_results is not None, "Don't forget results!"
    
    print("✅ Exercise 3 passed!")
    print("\nStrategy Performance:")
    print(strategy_results)

test_exercise_3()

## Summary

### Key Takeaways

1. **Market regimes** have distinct return and volatility characteristics
2. **HMMs automatically detect** regime structure from data
3. **Feature engineering** improves regime discrimination
4. **Regime persistence** enables tactical strategies
5. **Validation** against known events builds confidence

### Practical Insights

- Bull markets: ~70-80% of time, modest gains
- Bear markets: ~5-15% of time, sharp losses
- Sideways: ~10-20% of time, low returns
- Transitions: Often Bull → Sideways → Bear → Bull

### Next Steps

- **Module 4.2**: Volatility regime modeling (VIX)
- **Module 4.3**: Advanced strategy backtesting

---